# Conditional TimeGAN - Baseline (BCE)

Baseline implementation of conditional TimeGAN for synthetic CGM generation,
using the standard BCE adversarial objective.
The generator is conditioned on basal insulin, bolus insulin and carbohydrate intake;
CGM is the output channel.


## 0. Imports and reproducibility

In [ ]:
import os
import json
import random
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy import stats, signal
from scipy.stats import wasserstein_distance, ks_2samp
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")


## 1. Run configuration

In [ ]:
# ── Identification ─────────────────────────────────────────────────────────────
PATIENT_ID = "588"
RUN_NAME   = "baseline_bce"

# ── Paths ──────────────────────────────────────────────────────────────────────
INPUT_FOLDER   = Path("DatasetsProcessed") / f"patient_{PATIENT_ID}"
OUTPUT_BASE    = Path("ResultadosGenerativos") / "TimeGAN_condicional" / f"patient_{PATIENT_ID}" / f"run_{RUN_NAME}"
RAW_FOLDER     = OUTPUT_BASE / "raw"
FIGURES_FOLDER = OUTPUT_BASE / "figuras"
MODEL_FOLDER   = OUTPUT_BASE / "modelo"

# ── Data ───────────────────────────────────────────────────────────────────────
CGM_CHANNEL    = 0
INPUT_CHANNELS = [1, 2, 3]
ALL_CHANNELS   = [0, 1, 2, 3]
WINDOW_LENGTH  = 96

# ── Architecture ───────────────────────────────────────────────────────────────
HIDDEN_DIM = 32
NUM_LAYERS = 3
NOISE_DIM  = 32

# ── Training ───────────────────────────────────────────────────────────────────
BATCH_SIZE    = 128
LR            = 1e-3
GAMMA         = 0.1
EPOCHS_AE     = 200
EPOCHS_SUP    = 100
EPOCHS_JOINT  = 300
D_STEPS_PER_G = 2

# ── Generation ─────────────────────────────────────────────────────────────────
N_SYNTHETIC = 500

# ── Setup output folders ───────────────────────────────────────────────────────
if OUTPUT_BASE.exists() and any(OUTPUT_BASE.iterdir()):
    raise FileExistsError(
        f"Folder {OUTPUT_BASE} already exists. "
        f"Change RUN_NAME or delete it manually before re-running."
    )

for d in (OUTPUT_BASE, RAW_FOLDER, FIGURES_FOLDER, MODEL_FOLDER):
    d.mkdir(parents=True, exist_ok=True)

config = {
    "patient_id": PATIENT_ID,
    "run_name": RUN_NAME,
    "model_type": "standard",
    "seed": SEED,
    "device": str(DEVICE),
    "window_length": WINDOW_LENGTH,
    "cgm_channel": CGM_CHANNEL,
    "input_channels": INPUT_CHANNELS,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "noise_dim": NOISE_DIM,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "gamma": GAMMA,
    "epochs_ae": EPOCHS_AE,
    "epochs_sup": EPOCHS_SUP,
    "epochs_joint": EPOCHS_JOINT,
    "d_steps": D_STEPS_PER_G,
    "n_synthetic": N_SYNTHETIC,
}
with open(OUTPUT_BASE / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Run configuration:")
for k, v in config.items():
    print(f"  {k}: {v}")


## 2. Data loading

In [ ]:
windows_path  = INPUT_FOLDER / "windows.npy"
metadata_path = INPUT_FOLDER / "metadata.json"

assert windows_path.exists(),  f"Not found: {windows_path}"
assert metadata_path.exists(), f"Not found: {metadata_path}"

data_full = np.load(windows_path).astype(np.float32)
with open(metadata_path) as f:
    meta = json.load(f)

N_orig, L_orig, C = data_full.shape
print(f"Dataset loaded: N={N_orig}, L={L_orig}, C={C}")
print(f"Channels: {meta['channels']}")

if WINDOW_LENGTH > L_orig:
    raise ValueError(f"WINDOW_LENGTH={WINDOW_LENGTH} > L_orig={L_orig}.")
elif WINDOW_LENGTH < L_orig:
    data_full = data_full[:, :WINDOW_LENGTH, :]

L = WINDOW_LENGTH

X_cond = data_full[:, :, INPUT_CHANNELS]
X_cgm  = data_full[:, :, [CGM_CHANNEL]]
X_all  = data_full[:, :, ALL_CHANNELS]

n_input  = X_cond.shape[2]
n_output = 1

phys = meta["physiological_limits"]

def denorm(x_norm, key):
    """Denormalises from [-1, 1] to original units."""
    lo, hi = phys[key]
    return (x_norm + 1.0) / 2.0 * (hi - lo) + lo

CGM_LO, CGM_HI = phys["cgm"]

X_cond_t = torch.tensor(X_cond, dtype=torch.float32)
X_cgm_t  = torch.tensor(X_cgm,  dtype=torch.float32)
dataset    = TensorDataset(X_cond_t, X_cgm_t)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

print(f"DataLoader: {len(dataloader)} batches of size {BATCH_SIZE}")

cgm_real_mgdl = denorm(X_cgm.flatten(), "cgm")
print(f"\nReal CGM (mg/dL):")
print(f"  Mean: {cgm_real_mgdl.mean():.1f}")
print(f"  Std:  {cgm_real_mgdl.std():.1f}")
print(f"  TIR (70-180): {((cgm_real_mgdl >= 70) & (cgm_real_mgdl <= 180)).mean()*100:.1f}%")


## 3. Conditional TimeGAN architecture

Five GRU-based networks operating around a shared deterministic latent space:
- **Embedder E**: CGM → H_real
- **Recovery R**: H → CGM
- **Generator G**: [z_noise, C_cond] → H_fake
- **Supervisor S**: H_t → H_{t+1}
- **Discriminator D**: H → logit


In [ ]:
class GRUNet(nn.Module):
    """Reusable GRU block with linear output projection."""
    def __init__(self, input_dim, hidden_dim, output_dim,
                 num_layers=3, dropout=0.0, activation="tanh"):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_dim, output_dim)
        self.act = {"tanh": nn.Tanh(), "sigmoid": nn.Sigmoid(), "none": nn.Identity()}[activation]

    def forward(self, x):
        out, _ = self.gru(x)
        return self.act(self.fc(out))


class TimeGANConditional(nn.Module):
    """Conditional TimeGAN with deterministic autoencoder and BCE adversarial loss."""
    def __init__(self, n_input, n_output, hidden_dim, noise_dim, num_layers):
        super().__init__()
        self.noise_dim = noise_dim

        self.embedder      = GRUNet(n_output,            hidden_dim, hidden_dim, num_layers, activation="sigmoid")
        self.recovery      = GRUNet(hidden_dim,           hidden_dim, n_output,  num_layers, activation="tanh")
        self.generator     = GRUNet(noise_dim + n_input,  hidden_dim, hidden_dim, num_layers, activation="sigmoid")
        self.supervisor    = GRUNet(hidden_dim,           hidden_dim, hidden_dim,
                                    max(1, num_layers - 1), activation="sigmoid")
        self.discriminator = GRUNet(hidden_dim,           hidden_dim, 1,
                                    max(1, num_layers - 1), activation="none")

    def encode(self, x):       return self.embedder(x)
    def decode(self, h):       return self.recovery(h)
    def generate(self, z, c):  return self.generator(torch.cat([z, c], dim=-1))
    def supervise(self, h):    return self.supervisor(h)
    def discriminate(self, h): return self.discriminator(h)

    @torch.no_grad()
    def sample(self, c, n=None):
        self.eval()
        if n is not None and c.shape[0] == 1:
            c = c.repeat(n, 1, 1)
        B, L, _ = c.shape
        z = torch.randn(B, L, self.noise_dim, device=c.device)
        return self.decode(self.generate(z, c))


model = TimeGANConditional(
    n_input=n_input, n_output=n_output,
    hidden_dim=HIDDEN_DIM, noise_dim=NOISE_DIM, num_layers=NUM_LAYERS,
).to(DEVICE)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

print("Conditional TimeGAN architecture:")
for name, net in [("Embedder", model.embedder), ("Recovery", model.recovery),
                   ("Generator", model.generator), ("Supervisor", model.supervisor),
                   ("Discriminator", model.discriminator)]:
    print(f"  {name}: {count_params(net):,} params")
print(f"  TOTAL: {count_params(model):,} params")


## 4. Optimisers and loss functions

In [ ]:
opt_ae   = torch.optim.Adam(list(model.embedder.parameters()) + list(model.recovery.parameters()), lr=LR)
opt_sup  = torch.optim.Adam(model.supervisor.parameters(), lr=LR)
opt_gen  = torch.optim.Adam(list(model.generator.parameters()) + list(model.supervisor.parameters()), lr=LR)
opt_disc = torch.optim.Adam(model.discriminator.parameters(), lr=LR)

mse_loss    = nn.MSELoss()
adv_loss_fn = nn.BCEWithLogitsLoss()

def disc_loss(d_real, d_fake):
    """BCE discriminator loss."""
    return adv_loss_fn(d_real, torch.ones_like(d_real)) + adv_loss_fn(d_fake, torch.zeros_like(d_fake))

def gen_adv_loss(d_fake):
    """BCE generator adversarial loss."""
    return adv_loss_fn(d_fake, torch.ones_like(d_fake))

history = {"epoch": [], "phase": [], "loss_ae_recon": [],
           "loss_sup": [], "loss_gen_adv": [], "loss_gen_sup": [], "loss_disc": []}

def log_loss(epoch, phase, **kwargs):
    history["epoch"].append(epoch)
    history["phase"].append(phase)
    for key in ["loss_ae_recon", "loss_sup", "loss_gen_adv", "loss_gen_sup", "loss_disc"]:
        history[key].append(kwargs.get(key, float("nan")))

print("Optimisers and loss functions configured (BCE).")


## 5. Training - Phase 1: Autoencoder

In [ ]:
print(f"{'='*60}\n Phase 1: Autoencoder ({EPOCHS_AE} epochs)\n{'='*60}")
model.train()
LOG_INTERVAL = max(1, EPOCHS_AE // 10)

for epoch in range(1, EPOCHS_AE + 1):
    epoch_loss = 0.0
    for batch_cond, batch_cgm in dataloader:
        batch_cgm = batch_cgm.to(DEVICE)
        h_real  = model.encode(batch_cgm)
        cgm_rec = model.decode(h_real)
        loss_ae = mse_loss(cgm_rec, batch_cgm)
        opt_ae.zero_grad()
        loss_ae.backward()
        torch.nn.utils.clip_grad_norm_(
            list(model.embedder.parameters()) + list(model.recovery.parameters()), max_norm=5.0)
        opt_ae.step()
        epoch_loss += loss_ae.item()
    avg_loss = epoch_loss / len(dataloader)
    log_loss(epoch, "ae", loss_ae_recon=avg_loss)
    if epoch % LOG_INTERVAL == 0 or epoch == 1:
        print(f"  Epoch {epoch:4d}/{EPOCHS_AE} | AE loss: {avg_loss:.6f}")

print(f"\nPhase 1 complete. Final AE loss: {avg_loss:.6f}")


## 6. Training - Phase 2: Supervisor

In [ ]:
print(f"{'='*60}\n Phase 2: Supervisor ({EPOCHS_SUP} epochs)\n{'='*60}")
model.train()
LOG_INTERVAL = max(1, EPOCHS_SUP // 10)

for epoch in range(1, EPOCHS_SUP + 1):
    epoch_loss = 0.0
    for batch_cond, batch_cgm in dataloader:
        batch_cgm = batch_cgm.to(DEVICE)
        with torch.no_grad():
            h_real = model.encode(batch_cgm)
        h_sup    = model.supervise(h_real[:, :-1, :])
        loss_sup = mse_loss(h_sup, h_real[:, 1:, :])
        opt_sup.zero_grad()
        loss_sup.backward()
        torch.nn.utils.clip_grad_norm_(model.supervisor.parameters(), max_norm=5.0)
        opt_sup.step()
        epoch_loss += loss_sup.item()
    avg_loss = epoch_loss / len(dataloader)
    log_loss(epoch, "sup", loss_sup=avg_loss)
    if epoch % LOG_INTERVAL == 0 or epoch == 1:
        print(f"  Epoch {epoch:4d}/{EPOCHS_SUP} | SUP loss: {avg_loss:.6f}")

print(f"\nPhase 2 complete. Final SUP loss: {avg_loss:.6f}")


## 7. Training - Phase 3: Joint adversarial training

In [ ]:
print(f"{'='*60}\n Phase 3: Joint adversarial ({EPOCHS_JOINT} epochs) — BCE\n{'='*60}")
model.train()
LOG_INTERVAL = max(1, EPOCHS_JOINT // 20)

for epoch in range(1, EPOCHS_JOINT + 1):
    g_losses_adv, g_losses_sup, d_losses = [], [], []

    for batch_cond, batch_cgm in dataloader:
        batch_cond = batch_cond.to(DEVICE)
        batch_cgm  = batch_cgm.to(DEVICE)
        B = batch_cgm.shape[0]

        # Discriminator step
        for _ in range(D_STEPS_PER_G):
            z      = torch.randn(B, L, NOISE_DIM, device=DEVICE)
            h_real = model.encode(batch_cgm)
            h_fake = model.generate(z, batch_cond)
            loss_d = disc_loss(model.discriminate(h_real), model.discriminate(h_fake.detach()))
            opt_disc.zero_grad()
            loss_d.backward()
            torch.nn.utils.clip_grad_norm_(model.discriminator.parameters(), max_norm=5.0)
            opt_disc.step()
            d_losses.append(loss_d.item())

        # Generator step
        z      = torch.randn(B, L, NOISE_DIM, device=DEVICE)
        h_fake = model.generate(z, batch_cond)
        l_adv  = gen_adv_loss(model.discriminate(h_fake))
        h_sup_fake = model.supervise(h_fake[:, :-1, :])
        l_sup  = mse_loss(h_sup_fake, h_fake[:, 1:, :].detach())
        h_real = model.encode(batch_cgm)
        l_recon = mse_loss(model.decode(h_real), batch_cgm)
        loss_g = l_adv + GAMMA * l_sup + l_recon

        opt_gen.zero_grad()
        opt_ae.zero_grad()
        loss_g.backward()
        torch.nn.utils.clip_grad_norm_(
            list(model.generator.parameters()) + list(model.supervisor.parameters()), max_norm=5.0)
        opt_gen.step()
        opt_ae.step()
        g_losses_adv.append(l_adv.item())
        g_losses_sup.append(l_sup.item())

    avg_g_adv = np.mean(g_losses_adv)
    avg_g_sup = np.mean(g_losses_sup)
    avg_d     = np.mean(d_losses)
    log_loss(epoch, "joint", loss_gen_adv=avg_g_adv, loss_gen_sup=avg_g_sup, loss_disc=avg_d)

    if epoch % LOG_INTERVAL == 0 or epoch == 1:
        print(f"  Epoch {epoch:4d}/{EPOCHS_JOINT} | G_adv={avg_g_adv:.4f}  G_sup={avg_g_sup:.4f}  D={avg_d:.4f}")

print(f"\nPhase 3 complete.")

for name, net in [("embedder", model.embedder), ("recovery", model.recovery),
                   ("generator", model.generator), ("supervisor", model.supervisor),
                   ("discriminator", model.discriminator)]:
    torch.save(net.state_dict(), MODEL_FOLDER / f"{name}.pt")
print(f"Weights saved to {MODEL_FOLDER}")

pd.DataFrame(history).to_csv(RAW_FOLDER / "losses.csv", index=False)
print(f"Loss history saved to {RAW_FOLDER / 'losses.csv'}")


## 8. Synthetic sample generation

In [ ]:
model.eval()
idx_cond = np.random.choice(len(X_cond), size=N_SYNTHETIC, replace=True)
c_sample  = torch.tensor(X_cond[idx_cond], dtype=torch.float32).to(DEVICE)

with torch.no_grad():
    cgm_fake_norm = model.sample(c_sample).cpu().numpy()

cgm_synt_mgdl = denorm(cgm_fake_norm.flatten(), "cgm")
cgm_real_mgdl = denorm(X_cgm.flatten(), "cgm")

print(f"Synthetic: {cgm_fake_norm.shape}")
print(f"Synthetic CGM (mg/dL): mean={cgm_synt_mgdl.mean():.1f}, std={cgm_synt_mgdl.std():.1f}")

np.save(RAW_FOLDER / "marginal_real.npy", cgm_real_mgdl)
np.save(RAW_FOLDER / "marginal_synt.npy", cgm_synt_mgdl)
np.save(RAW_FOLDER / "windows_synt_mgdl.npy",
        denorm(cgm_fake_norm, "cgm").reshape(N_SYNTHETIC, L, 1))
np.save(RAW_FOLDER / "windows_real_mgdl.npy",
        denorm(X_cgm, "cgm"))
